In [2]:
from nrem_analysis.constant import RAW_DIR, PROCESSED_DIR, FIGURES_DIR, MOUSE_IDS_DUAL

from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pynapple as nap

## Cross-correlograms of turn cells

In [19]:
data = {}
states = ["wake", "nrem", "rem"]

for mouse_id in MOUSE_IDS_DUAL:
    sleep = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "sleep.npz")
    turn_units = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "turn_units.npz")
    print(f"Processing mouse {mouse_id} ({len(turn_units)} turn units)")
    data[mouse_id] = {}
    for state in states:
        if state == "nrem":
            binsize = 20
            windowsize = 400
        else:
            binsize = 200
            windowsize = 4001

        epochs = sleep[sleep['state'] == state]

        result = nap.compute_crosscorrelogram(
            group=turn_units,
            binsize=binsize,
            windowsize=windowsize,
            ep=epochs,
            time_units="ms",
            norm=False,
        )

        print(f"Computed CCG for state {state} ({epochs.tot_length().item()} s) with shape {result.shape}")
        data[mouse_id][state] = result

Processing mouse 99b (17 turn units)
Computed CCG for state wake (27985.0 s) with shape (41, 136)
Computed CCG for state nrem (11796.0 s) with shape (41, 136)
Computed CCG for state rem (944.0 s) with shape (41, 136)
Processing mouse 100b (30 turn units)
Computed CCG for state wake (27233.0 s) with shape (41, 435)
Computed CCG for state nrem (13810.0 s) with shape (41, 435)
Computed CCG for state rem (643.0 s) with shape (41, 435)
Processing mouse 102b (25 turn units)
Computed CCG for state wake (20633.0 s) with shape (41, 300)
Computed CCG for state nrem (14230.0 s) with shape (41, 300)
Computed CCG for state rem (1356.0 s) with shape (41, 300)
Processing mouse 103c (7 turn units)
Computed CCG for state wake (18485.0 s) with shape (41, 21)
Computed CCG for state nrem (15963.0 s) with shape (41, 21)
Computed CCG for state rem (1729.0 s) with shape (41, 21)
Processing mouse 106b (31 turn units)
Computed CCG for state wake (26973.0 s) with shape (41, 465)
Computed CCG for state nrem (140

### Per mouse CCG

In [ ]:
save_dir = FIGURES_DIR / "turn_ccg"
save_dir.mkdir(parents=True, exist_ok=True)

for mouse_id in data.keys():
    mouse_data = data[mouse_id]
    fig, ax = plt.subplots(2, 3, figsize=(18, 6), constrained_layout=True, gridspec_kw={'height_ratios': [1, 0.05], 'wspace': 0.05, 'hspace': 0.1})
    
    # determine pair order
    wake_order = np.argsort(mouse_data['wake'].to_numpy().max(axis=0))

    for i, (state, ccg) in enumerate(mouse_data.items()):
        ccg, t = ccg.to_numpy(), ccg.index.values
        # normalize
        ccg = ccg - np.median(ccg, axis=0)
        im = ax[0, i].imshow(
            ccg[:, wake_order].T,
            aspect='auto',
            origin='lower',
            cmap='hot',
            interpolation='none',
            extent=[t[0], t[-1], 0, ccg.shape[1]]
            )
        ax[0, i].set_xlabel('Time (s)')
        ax[0, i].set_title(state)
        cb = fig.colorbar(im, cax=ax[1, i], orientation='horizontal')

    ax[0, 0].set_ylabel('Turn cell pair ID')
    fig.suptitle(f"{mouse_id}", fontsize=16)
    plt.savefig(save_dir / f"{mouse_id}.png", dpi=320, bbox_inches='tight')
    plt.close(fig)

### All mice CCG

In [15]:
concat_data = {'wake': [], 'nrem': [], 'rem': []}
count_pairs = {'wake': 0, 'nrem': 0, 'rem': 0}

for mouse_id in data.keys():
    mouse_data = data[mouse_id]
    for state in states:
        concat_data[state].append(mouse_data[state])
        count_pairs[state] += mouse_data[state].shape[1]

for state in states:
    concat_data[state] = pd.concat(concat_data[state], axis=1)
    print(f"Concatenated {count_pairs[state]} pairs for state {state} with shape {concat_data[state].shape}")

Concatenated 2144 pairs for state wake with shape (41, 2144)
Concatenated 2144 pairs for state nrem with shape (41, 2144)
Concatenated 2144 pairs for state rem with shape (41, 2144)


In [20]:
fig, ax = plt.subplots(2, 3, figsize=(18, 6), constrained_layout=True, gridspec_kw={'height_ratios': [1, 0.05], 'wspace': 0.05, 'hspace': 0.1})
    
# determine pair order
wake_order = np.argsort(concat_data['wake'].to_numpy().max(axis=0))

for i, (state, ccg) in enumerate(concat_data.items()):
    ccg, t = ccg.to_numpy(), ccg.index.values
    # normalize
    ccg = ccg - np.median(ccg, axis=0)
    im = ax[0, i].imshow(
        ccg[:, wake_order].T,
        aspect='auto',
        origin='lower',
        cmap='hot',
        interpolation='none',
        extent=[t[0], t[-1], 0, ccg.shape[1]]
        )
    ax[0, i].set_xlabel('Time (s)')
    ax[0, i].set_title(state)
    cb = fig.colorbar(im, cax=ax[1, i], orientation='horizontal')

ax[0, 0].set_ylabel('Turn cell pair ID')
fig.suptitle(f"All mice (n={len(data)})", fontsize=16)
plt.savefig(save_dir / f"all.png", dpi=320, bbox_inches='tight')
plt.close(fig)